In [1]:
pip install streamlit pandas numpy scikit-learn joblib matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import zipfile, glob, os
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd()
ZIP_PATH = PROJECT_DIR / "har70.zip"
DATA_DIR = PROJECT_DIR / "har70plus"

print("Notebook folder:", PROJECT_DIR)
print("har70.zip found:", ZIP_PATH.exists())

Notebook folder: C:\Users\eshit
har70.zip found: True


In [8]:
if ZIP_PATH.exists() and not DATA_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(PROJECT_DIR)

csv_files = sorted(glob.glob(str(DATA_DIR / "*.csv")))
print("Participant CSV files:", len(csv_files))
print(csv_files[:5])

Participant CSV files: 18
['C:\\Users\\eshit\\har70plus\\501.csv', 'C:\\Users\\eshit\\har70plus\\502.csv', 'C:\\Users\\eshit\\har70plus\\503.csv', 'C:\\Users\\eshit\\har70plus\\504.csv', 'C:\\Users\\eshit\\har70plus\\505.csv']


In [5]:
sample = pd.read_csv(csv_files[0])
display(sample.head())
print("Shape:", sample.shape)
print("Columns:", sample.columns.tolist())
print("Labels:")
display(sample["label"].value_counts().sort_index())

,timestamp,back_x,back_y,back_z,thigh_x,thigh_y,thigh_z,label
0,2021-03-24 14:42:03.839,-0.999023,-0.063477,0.140625,-0.980469,-0.112061,-0.048096,6
1,2021-03-24 14:42:03.859,-0.980225,-0.079346,0.140625,-0.961182,-0.121582,-0.051758,6
2,2021-03-24 14:42:03.880,-0.950195,-0.076416,0.140625,-0.949463,-0.080566,-0.067139,6
3,2021-03-24 14:42:03.900,-0.954834,-0.059082,0.140381,-0.957520,-0.046143,-0.050781,6
4,2021-03-24 14:42:03.920,-0.972412,-0.042969,0.142822,-0.977051,-0.023682,-0.026611,6


Shape: (103860, 8)
Columns: ['timestamp', 'back_x', 'back_y', 'back_z', 'thigh_x', 'thigh_y', 'thigh_z', 'label']
Labels:


label
1    62872
3     1876
4       92
5      504
6    14376
7    14562
8     9578
Name: count, dtype: int64

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
Feature_Cols = ["back_x","back_y","back_z","thigh_x","thigh_y","thigh_z"]
Window = 100
def make_features(filepaths, window=Window):
    X, y, groups = [], [], []
    for f in filepaths:
        d = pd.read_csv(f).dropna(subset= Feature_Cols + ["label"])
        for label, g in d.groupby("label", sort=False):
            arr = g[Feature_Cols].to_numpy()
            n = (len(arr) // window) * window
            if n == 0:
                continue
            arr = arr[:n].reshape(-1, window, len(Feature_Cols))
            feats = np.concatenate([arr.mean(axis=1),arr.std(axis=1),arr.min(axis=1),arr.max(axis=1)], axis=1)
            X.append(feats)
            y.extend([int(label)] * len(feats))
            groups.extend([Path(f).stem] * len(feats))
    return np.vstack(X), np.array(y), np.array(groups)
print("Feature function ready.")

Feature function ready.


In [9]:
test_ids = {"501", "502", "503", "504"}
train_files = [f for f in csv_files if Path(f).stem not in test_ids]
test_files = [f for f in csv_files if Path(f).stem in test_ids]
X_train, y_train, _ = make_features(train_files)
X_test, y_test, _ = make_features(test_files)
print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (17529, 24)
Testing: (5011, 24)


In [10]:
model = RandomForestClassifier(n_estimators = 120, random_state = 42, n_jobs = -1,class_weight = "balanced_subsample")
model.fit(X_train, y_train)
print("Training complete")

Training complete


In [11]:
pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, zero_division=0))

Accuracy: 0.9607
              precision    recall  f1-score   support

           1       0.95      1.00      0.97      2512
           3       0.69      0.15      0.25       167
           4       0.00      0.00      0.00         2
           5       0.00      0.00      0.00         7
           6       0.96      0.97      0.96      1040
           7       0.99      1.00      1.00       850
           8       1.00      1.00      1.00       433

    accuracy                           0.96      5011
   macro avg       0.66      0.59      0.60      5011
weighted avg       0.95      0.96      0.95      5011



In [12]:
MODEL_PATH = PROJECT_DIR / "eldercare_activity_model.joblib"
joblib.dump(model, MODEL_PATH)
print("Saved model to:", MODEL_PATH)

Saved model to: C:\Users\eshit\eldercare_activity_model.joblib


In [13]:
LABEL_NAMES = {1:"Walking", 3:"Shuffling", 4:"Ascending stairs",5:"Descending stairs", 6:"Standing", 7:"Sitting", 8:"Lying"}
demo = pd.read_csv(test_files[0])
X_demo, _, _ = make_features([test_files[0]])
demo_pred = model.predict(X_demo[:10])
for i, p in enumerate(demo_pred, 1):
    print(f"Window {i}: {LABEL_NAMES.get(int(p), p)}")

Window 1: Standing
Window 2: Standing
Window 3: Standing
Window 4: Standing
Window 5: Standing
Window 6: Standing
Window 7: Standing
Window 8: Standing
Window 9: Standing
Window 10: Standing


In [20]:
import os

print(os.listdir(r"C:\Users\eshit"))

['.anaconda', '.android', '.cache', '.cassandra', '.conda', '.condarc', '.continuum', '.eclipse', '.ipynb_checkpoints', '.ipython', '.jupyter', '.kaggle', '.keras', '.matplotlib', '.Neo4jDesktop', '.p2', '.ssh', '.streamlit', '.VirtualBox', '.vscode', '.wdm', '13-03-2024 DATA STRUCTURES.ipynb', '22BTRAD014_LABWORK10.ipynb', '22BTRAD014_LABWORK11.ipynb', '22BTRAD014_LABWORK12.ipynb', '22BTRAD014_LABWORK6.ipynb', '22BTRAD014_LABWORK7.ipynb', '22BTRAD014_LABWORK8.ipynb', '22BTRAD014_LABWORK9.ipynb', '22BTRAD014_Lab_Work1.CSV', '22BTRAD014_Lab_Work1.CSV.xlsx', '22BTRAD014_Lab_Work2.csv', '22BTRAD014_Lab_Work3.csv', '22BTRAD014_Lab_Work5.csv', '22BTRAD014_LAB_WORK5.ipynb', '22BTRAD014_LAB_WORK_03.ipynb', '22BTRAD014_LAB_WORK_04.ipynb', '22BTRAD014_LAB_WORK_2.ipynb', '22BTRAD014_module 5 SEDA Coding.ipynb', '22TCSE544 -Data Streaming and Analysis.pdf', '3rd year Internship.ipynb', '6062401_20260625C.csv', 'actual_max_temp.png', 'actual_mean_temp.png', 'actual_min_temp.png', 'actual_precipita

In [21]:
import os

files = [
    "app.py",
    "eldercare_activity_model.joblib",
    "ElderCare_Assist.ipynb",
    "har70.zip",
    "har70plus"
]

for file in files:
    print(file, "->", os.path.exists(file))

app.py -> True
eldercare_activity_model.joblib -> True
ElderCare_Assist.ipynb -> True
har70.zip -> True
har70plus -> True


In [22]:
import pandas as pd

source_file = r"C:\Users\eshit\har70plus\501.csv"

df = pd.read_csv(source_file)

print("Shape:", df.shape)
print(df.head())

Shape: (103860, 8)
                 timestamp    back_x    back_y    back_z   thigh_x   thigh_y  \
0  2021-03-24 14:42:03.839 -0.999023 -0.063477  0.140625 -0.980469 -0.112061   
1  2021-03-24 14:42:03.859 -0.980225 -0.079346  0.140625 -0.961182 -0.121582   
2  2021-03-24 14:42:03.880 -0.950195 -0.076416  0.140625 -0.949463 -0.080566   
3  2021-03-24 14:42:03.900 -0.954834 -0.059082  0.140381 -0.957520 -0.046143   
4  2021-03-24 14:42:03.920 -0.972412 -0.042969  0.142822 -0.977051 -0.023682   

    thigh_z  label  
0 -0.048096      6  
1 -0.051758      6  
2 -0.067139      6  
3 -0.050781      6  
4 -0.026611      6  


In [23]:
demo = df.head(2000)

demo.to_csv(
    r"C:\Users\eshit\demo_sensor_data.csv",
    index=False
)

print("demo_sensor_data.csv created successfully!")

demo_sensor_data.csv created successfully!


In [24]:
import os

for file in [
    "app.py",
    "eldercare_activity_model.joblib",
    "demo_sensor_data.csv"
]:
    print(file, "->", os.path.exists(file))

app.py -> True
eldercare_activity_model.joblib -> True
demo_sensor_data.csv -> True


In [27]:
import pandas as pd

sensor_file = r"C:\Users\eshit\har70plus\501.csv"

df = pd.read_csv(sensor_file)

print(df.shape)
display(df.head())

(103860, 8)


,timestamp,back_x,back_y,back_z,thigh_x,thigh_y,thigh_z,label
0,2021-03-24 14:42:03.839,-0.999023,-0.063477,0.140625,-0.980469,-0.112061,-0.048096,6
1,2021-03-24 14:42:03.859,-0.980225,-0.079346,0.140625,-0.961182,-0.121582,-0.051758,6
2,2021-03-24 14:42:03.880,-0.950195,-0.076416,0.140625,-0.949463,-0.080566,-0.067139,6
3,2021-03-24 14:42:03.900,-0.954834,-0.059082,0.140381,-0.957520,-0.046143,-0.050781,6
4,2021-03-24 14:42:03.920,-0.972412,-0.042969,0.142822,-0.977051,-0.023682,-0.026611,6


In [28]:
demo_data = df.head(2000)

demo_data.to_csv(
    r"C:\Users\eshit\demo_sensor_data.csv",
    index=False
)

print("demo_sensor_data.csv created successfully!")

demo_sensor_data.csv created successfully!


In [29]:
import os

required_files = [
    "eldercare_activity_model.joblib",
    "demo_sensor_data.csv"
]

for file in required_files:
    print(f"{file}: {os.path.exists(file)}")

eldercare_activity_model.joblib: True
demo_sensor_data.csv: True


In [3]:
from pathlib import Path

APP_CODE = r'''
import streamlit as st
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

APP_DIR = Path(__file__).resolve().parent

MODEL_PATH = APP_DIR / "eldercare_activity_model.joblib"
DEMO_PATH = APP_DIR / "demo_sensor_data.csv"

FEATURE_COLS = [
    "back_x",
    "back_y",
    "back_z",
    "thigh_x",
    "thigh_y",
    "thigh_z"
]

LABEL_NAMES = {
    1: "Walking",
    3: "Shuffling",
    4: "Ascending stairs",
    5: "Descending stairs",
    6: "Standing",
    7: "Sitting",
    8: "Lying"
}

WINDOW = 100

st.set_page_config(
    page_title="ElderCare Assist",
    page_icon="👴",
    layout="wide"
)

st.title("👴 ElderCare Assist")
st.write("Elderly Assistance and Activity Monitoring System")

@st.cache_resource
def load_model():
    return joblib.load(MODEL_PATH)

def create_features(data):

    values = data[FEATURE_COLS].dropna().to_numpy()

    number_of_rows = (len(values) // WINDOW) * WINDOW

    if number_of_rows < WINDOW:
        raise ValueError(
            "At least 100 sensor rows are required."
        )

    values = values[:number_of_rows]

    values = values.reshape(
        -1,
        WINDOW,
        len(FEATURE_COLS)
    )

    features = np.concatenate([
        values.mean(axis=1),
        values.std(axis=1),
        values.min(axis=1),
        values.max(axis=1)
    ], axis=1)

    return features


# -------------------------
# SIDEBAR
# -------------------------

st.sidebar.title("Menu")

page = st.sidebar.radio(
    "Select",
    [
        "Dashboard",
        "Elderly Profile",
        "Assistance Request",
        "Reminders",
        "Emergency Help",
        "Activity Monitor"
    ]
)


# -------------------------
# DASHBOARD
# -------------------------

if page == "Dashboard":

    st.header("🏠 Dashboard")

    col1, col2, col3 = st.columns(3)

    col1.metric(
        "Elderly Users",
        "0"
    )

    col2.metric(
        "Pending Requests",
        "0"
    )

    col3.metric(
        "Reminders",
        "0"
    )

    st.divider()

    st.info(
        "Welcome to ElderCare Assist. "
        "This prototype provides elderly assistance features "
        "and machine-learning based activity recognition."
    )


# -------------------------
# PROFILE
# -------------------------

elif page == "Elderly Profile":

    st.header("👴 Elderly Profile")

    name = st.text_input("Name")

    age = st.number_input(
        "Age",
        min_value=1,
        max_value=120,
        value=70
    )

    phone = st.text_input("Phone")

    emergency_contact = st.text_input(
        "Emergency Contact"
    )

    if st.button("Save Profile"):

        if name.strip():

            st.success(
                f"Profile saved for {name}."
            )

        else:

            st.error(
                "Please enter the name."
            )


# -------------------------
# ASSISTANCE REQUEST
# -------------------------

elif page == "Assistance Request":

    st.header("🤝 Assistance Request")

    request_type = st.selectbox(
        "Request Type",
        [
            "Medicine",
            "Doctor Appointment",
            "Transportation",
            "Grocery",
            "Home Repair",
            "Other"
        ]
    )

    description = st.text_area(
        "Describe the assistance required"
    )

    priority = st.selectbox(
        "Priority",
        [
            "Low",
            "Medium",
            "High",
            "Emergency"
        ]
    )

    if st.button("Submit Request"):

        st.success(
            "Assistance request submitted successfully!"
        )


# -------------------------
# REMINDERS
# -------------------------

elif page == "Reminders":

    st.header("💊 Medicine & Appointment Reminders")

    reminder = st.text_input(
        "Reminder",
        placeholder="Medicine - 8 PM"
    )

    reminder_date = st.date_input(
        "Date"
    )

    reminder_time = st.time_input(
        "Time"
    )

    if st.button("Add Reminder"):

        if reminder.strip():

            st.success(
                f"Reminder added: {reminder}"
            )

        else:

            st.error(
                "Enter a reminder."
            )


# -------------------------
# EMERGENCY
# -------------------------

elif page == "Emergency Help":

    st.header("🆘 Emergency Help")

    st.warning(
        "This is a prototype. "
        "It does not automatically contact emergency services."
    )

    reason = st.selectbox(
        "Alert Type",
        [
            "Immediate Assistance",
            "Possible Accident",
            "Medical Assistance Needed",
            "Unable to Contact Person"
        ]
    )

    if st.button("🚨 Create Alert"):

        st.error(
            "Emergency alert recorded for caregiver review."
        )


# -------------------------
# ACTIVITY MONITOR
# -------------------------

elif page == "Activity Monitor":

    st.header("🤖 Activity Monitor")

    st.write(
        "HAR70+ sensor-based activity recognition"
    )

    option = st.radio(
        "Choose data source",
        [
            "Use Demo Data",
            "Upload CSV"
        ]
    )

    if option == "Use Demo Data":

        if not DEMO_PATH.exists():

            st.error(
                "demo_sensor_data.csv not found."
            )

            st.stop()

        data = pd.read_csv(
            DEMO_PATH
        )

    else:

        uploaded_file = st.file_uploader(
            "Upload HAR70+ CSV",
            type=["csv"]
        )

        if uploaded_file is None:

            st.info(
                "Please upload a CSV file."
            )

            st.stop()

        data = pd.read_csv(
            uploaded_file
        )

    st.write(
        "Number of sensor rows:",
        len(data)
    )

    st.dataframe(
        data.head(10)
    )

    if st.button(
        "Run Activity Prediction"
    ):

        try:

            model = load_model()

            features = create_features(
                data
            )

            predictions = model.predict(
                features
            )

            probabilities = model.predict_proba(
                features
            ).max(axis=1)

            results = pd.DataFrame({

                "Window":
                range(1, len(predictions) + 1),

                "Activity":
                [
                    LABEL_NAMES.get(
                        int(prediction),
                        str(prediction)
                    )
                    for prediction in predictions
                ],

                "Confidence (%)":
                np.round(
                    probabilities * 100,
                    2
                )

            })

            st.success(
                "Prediction completed!"
            )

            st.dataframe(
                results,
                use_container_width=True,
                hide_index=True
            )

            st.subheader(
                "Activity Summary"
            )

            st.bar_chart(
                results["Activity"].value_counts()
            )

        except Exception as error:

            st.error(
                f"Prediction failed: {error}"
            )


st.divider()

st.caption(
    "ElderCare Assist | Educational Prototype"
)
'''

app_path = Path(r"C:\Users\eshit\eldercare_app.py")

app_path.write_text(
    APP_CODE,
    encoding="utf-8"
)

print("Streamlit app created successfully!")
print(app_path)

Streamlit app created successfully!
C:\Users\eshit\eldercare_app.py


In [5]:
import os

os.chdir(r"C:\Users\eshit")

print("App:", os.path.exists("eldercare_full_app.py"))
print("Model:", os.path.exists("eldercare_activity_model.joblib"))
print("Demo data:", os.path.exists("demo_sensor_data.csv"))

App: False
Model: True
Demo data: True


In [7]:
import subprocess
import sys

process = subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    r"C:\Users\eshit\eldercare_full_app.py"
])

print("ElderCare Assist is starting...")

ElderCare Assist is starting...


In [9]:
import os

os.chdir(r"C:\Users\eshit")

print(os.getcwd())

C:\Users\eshit


In [13]:
import os

print("Full app:",
      os.path.exists(r"C:\Users\eshit\Downloads\eldercare_full_app (1).py"))

print("ML model:",
      os.path.exists(r"C:\Users\eshit\eldercare_activity_model.joblib"))

print("Demo data:",
      os.path.exists(r"C:\Users\eshit\demo_sensor_data.csv"))

Full app: True
ML model: True
Demo data: True


In [16]:
import subprocess
import sys

process = subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    r"C:\Users\eshit\eldercare_full_app (1).py"
])

print("ElderCare Assist is starting...")

ElderCare Assist is starting...


In [17]:
from pathlib import Path

APP_CODE = r'''
import streamlit as st
import pandas as pd
import numpy as np
import sqlite3
import joblib
from pathlib import Path
from datetime import datetime, date, time

# ============================================================
# ELDERCARE ASSIST V2
# ============================================================

APP_DIR = Path(__file__).resolve().parent

DB_PATH = APP_DIR / "eldercare.db"
MODEL_PATH = APP_DIR / "eldercare_activity_model.joblib"
DEMO_PATH = APP_DIR / "demo_sensor_data.csv"

FEATURE_COLS = [
    "back_x",
    "back_y",
    "back_z",
    "thigh_x",
    "thigh_y",
    "thigh_z"
]

LABEL_NAMES = {
    1: "Walking",
    2: "Running",
    3: "Shuffling",
    4: "Ascending stairs",
    5: "Descending stairs",
    6: "Standing",
    7: "Sitting",
    8: "Lying"
}

WINDOW = 100


# ============================================================
# DATABASE
# ============================================================

def get_db():
    return sqlite3.connect(
        DB_PATH,
        check_same_thread=False
    )


def create_tables():

    db = get_db()
    cur = db.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            age INTEGER,
            phone TEXT,
            emergency_contact TEXT,
            emergency_phone TEXT,
            medical_notes TEXT,
            created_at TEXT
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS caregivers (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            phone TEXT,
            email TEXT,
            relation TEXT,
            created_at TEXT
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS medicines (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER,
            medicine_name TEXT,
            dosage TEXT,
            frequency TEXT,
            medicine_time TEXT,
            start_date TEXT,
            end_date TEXT,
            status TEXT,
            notes TEXT
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS reminders (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER,
            title TEXT,
            reminder_date TEXT,
            reminder_time TEXT,
            notes TEXT,
            status TEXT,
            created_at TEXT
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS requests (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER,
            request_type TEXT,
            description TEXT,
            priority TEXT,
            status TEXT,
            created_at TEXT
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS alerts (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER,
            alert_type TEXT,
            description TEXT,
            status TEXT,
            created_at TEXT
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS activity_results (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER,
            activity TEXT,
            confidence REAL,
            window_no INTEGER,
            created_at TEXT
        )
    """)

    db.commit()
    db.close()


create_tables()


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_users():

    db = get_db()

    df = pd.read_sql_query(
        "SELECT * FROM users ORDER BY name",
        db
    )

    db.close()

    return df


def get_user_options():

    df = get_users()

    return {
        f"{row['name']} | ID {row['id']}":
        int(row["id"])
        for _, row in df.iterrows()
    }


def get_user_name(user_id):

    db = get_db()

    result = db.execute(
        "SELECT name FROM users WHERE id=?",
        (user_id,)
    ).fetchone()

    db.close()

    return result[0] if result else "Unknown"


@st.cache_resource
def load_model():

    return joblib.load(MODEL_PATH)


def create_features(data):

    missing = [
        col for col in FEATURE_COLS
        if col not in data.columns
    ]

    if missing:

        raise ValueError(
            "Missing columns: "
            + ", ".join(missing)
        )

    values = (
        data[FEATURE_COLS]
        .apply(pd.to_numeric, errors="coerce")
        .dropna()
        .to_numpy()
    )

    usable = (
        len(values) // WINDOW
    ) * WINDOW

    if usable < WINDOW:

        raise ValueError(
            "At least 100 valid sensor rows are required."
        )

    values = values[:usable]

    windows = values.reshape(
        -1,
        WINDOW,
        len(FEATURE_COLS)
    )

    features = np.concatenate(
        [
            windows.mean(axis=1),
            windows.std(axis=1),
            windows.min(axis=1),
            windows.max(axis=1)
        ],
        axis=1
    )

    return features


def save_activity_results(user_id, result):

    db = get_db()

    for _, row in result.iterrows():

        db.execute(
            """
            INSERT INTO activity_results
            (user_id, activity, confidence,
             window_no, created_at)
            VALUES (?, ?, ?, ?, ?)
            """,
            (
                user_id,
                row["Activity"],
                float(row["Confidence (%)"]),
                int(row["Window"]),
                datetime.now().strftime(
                    "%Y-%m-%d %H:%M:%S"
                )
            )
        )

    db.commit()
    db.close()


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="ElderCare Assist",
    page_icon="👴",
    layout="wide"
)


# ============================================================
# TITLE
# ============================================================

st.title("👴 ElderCare Assist")

st.caption(
    "A smart elderly assistance platform using "
    "Python, Streamlit, SQLite and Machine Learning."
)


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.title("Navigation")

page = st.sidebar.radio(
    "Select Module",
    [
        "🏠 Dashboard",
        "👴 Elderly Profile",
        "👨‍⚕️ Caregiver",
        "💊 Medicine",
        "🔔 Reminders",
        "🤝 Assistance Requests",
        "🆘 Emergency Help",
        "🤖 Activity Monitor",
        "📊 Activity History",
        "📋 Request Management"
    ]
)


# ============================================================
# DASHBOARD
# ============================================================

if page == "🏠 Dashboard":

    st.header("🏠 Dashboard")

    db = get_db()

    total_users = db.execute(
        "SELECT COUNT(*) FROM users"
    ).fetchone()[0]

    total_caregivers = db.execute(
        "SELECT COUNT(*) FROM caregivers"
    ).fetchone()[0]

    active_medicines = db.execute(
        """
        SELECT COUNT(*)
        FROM medicines
        WHERE status='Active'
        """
    ).fetchone()[0]

    pending_reminders = db.execute(
        """
        SELECT COUNT(*)
        FROM reminders
        WHERE status='Pending'
        """
    ).fetchone()[0]

    open_requests = db.execute(
        """
        SELECT COUNT(*)
        FROM requests
        WHERE status NOT IN
        ('Resolved','Closed')
        """
    ).fetchone()[0]

    emergency_alerts = db.execute(
        """
        SELECT COUNT(*)
        FROM alerts
        WHERE status='New'
        """
    ).fetchone()[0]

    db.close()

    a, b, c, d = st.columns(4)

    a.metric(
        "👴 Elderly Users",
        total_users
    )

    b.metric(
        "👨‍⚕️ Caregivers",
        total_caregivers
    )

    c.metric(
        "💊 Active Medicines",
        active_medicines
    )

    d.metric(
        "🤝 Open Requests",
        open_requests
    )

    a, b = st.columns(2)

    a.metric(
        "🔔 Pending Reminders",
        pending_reminders
    )

    b.metric(
        "🆘 New Emergency Alerts",
        emergency_alerts
    )

    st.divider()

    st.subheader("Today's Reminders")

    db = get_db()

    reminders = pd.read_sql_query(
        """
        SELECT
            r.id,
            u.name AS person,
            r.title,
            r.reminder_time,
            r.status
        FROM reminders r
        LEFT JOIN users u
        ON r.user_id=u.id
        WHERE r.reminder_date=?
        ORDER BY r.reminder_time
        """,
        db,
        params=(str(date.today()),)
    )

    db.close()

    if reminders.empty:

        st.info(
            "No reminders scheduled for today."
        )

    else:

        st.dataframe(
            reminders,
            hide_index=True,
            use_container_width=True
        )


# ============================================================
# ELDERLY PROFILE
# ============================================================

elif page == "👴 Elderly Profile":

    st.header("👴 Elderly Profile")

    with st.form("profile_form"):

        name = st.text_input(
            "Full Name *"
        )

        col1, col2 = st.columns(2)

        with col1:

            age = st.number_input(
                "Age",
                min_value=1,
                max_value=120,
                value=65
            )

            phone = st.text_input(
                "Phone"
            )

        with col2:

            emergency_contact = st.text_input(
                "Emergency Contact"
            )

            emergency_phone = st.text_input(
                "Emergency Contact Phone"
            )

        medical_notes = st.text_area(
            "Medical / Important Notes"
        )

        submit = st.form_submit_button(
            "💾 Save Profile"
        )

        if submit:

            if not name.strip():

                st.error(
                    "Please enter a name."
                )

            else:

                db = get_db()

                db.execute(
                    """
                    INSERT INTO users
                    (name, age, phone,
                     emergency_contact,
                     emergency_phone,
                     medical_notes,
                     created_at)
                    VALUES (?, ?, ?, ?, ?, ?, ?)
                    """,
                    (
                        name,
                        age,
                        phone,
                        emergency_contact,
                        emergency_phone,
                        medical_notes,
                        datetime.now().strftime(
                            "%Y-%m-%d %H:%M:%S"
                        )
                    )
                )

                db.commit()
                db.close()

                st.success(
                    "Elderly profile saved!"
                )

    st.divider()

    st.subheader("Existing Profiles")

    users = get_users()

    if users.empty:

        st.info(
            "No elderly profiles yet."
        )

    else:

        st.dataframe(
            users,
            hide_index=True,
            use_container_width=True
        )


# ============================================================
# CAREGIVER
# ============================================================

elif page == "👨‍⚕️ Caregiver":

    st.header("👨‍⚕️ Caregiver Management")

    with st.form("caregiver_form"):

        name = st.text_input(
            "Caregiver Name *"
        )

        relation = st.selectbox(
            "Relationship",
            [
                "Son",
                "Daughter",
                "Spouse",
                "Relative",
                "Professional Caregiver",
                "Other"
            ]
        )

        phone = st.text_input(
            "Phone"
        )

        email = st.text_input(
            "Email"
        )

        submit = st.form_submit_button(
            "➕ Add Caregiver"
        )

        if submit:

            if not name.strip():

                st.error(
                    "Please enter caregiver name."
                )

            else:

                db = get_db()

                db.execute(
                    """
                    INSERT INTO caregivers
                    (name, phone, email,
                     relation, created_at)
                    VALUES (?, ?, ?, ?, ?)
                    """,
                    (
                        name,
                        phone,
                        email,
                        relation,
                        datetime.now().strftime(
                            "%Y-%m-%d %H:%M:%S"
                        )
                    )
                )

                db.commit()
                db.close()

                st.success(
                    "Caregiver added!"
                )

    st.divider()

    db = get_db()

    caregivers = pd.read_sql_query(
        """
        SELECT *
        FROM caregivers
        ORDER BY name
        """,
        db
    )

    db.close()

    st.dataframe(
        caregivers,
        hide_index=True,
        use_container_width=True
    )


# ============================================================
# MEDICINE
# ============================================================

elif page == "💊 Medicine":

    st.header("💊 Medicine Management")

    users = get_user_options()

    if not users:

        st.warning(
            "Create an elderly profile first."
        )

    else:

        with st.form("medicine_form"):

            person = st.selectbox(
                "Elderly Person",
                list(users.keys())
            )

            medicine_name = st.text_input(
                "Medicine Name *"
            )

            dosage = st.text_input(
                "Dosage",
                placeholder="5 mg"
            )

            frequency = st.selectbox(
                "Frequency",
                [
                    "Once daily",
                    "Twice daily",
                    "Three times daily",
                    "Weekly",
                    "As prescribed"
                ]
            )

            medicine_time = st.time_input(
                "Medicine Time",
                time(20, 0)
            )

            start_date = st.date_input(
                "Start Date",
                date.today()
            )

            end_date = st.date_input(
                "End Date",
                date.today()
            )

            notes = st.text_area(
                "Notes"
            )

            submit = st.form_submit_button(
                "➕ Add Medicine"
            )

            if submit:

                if not medicine_name.strip():

                    st.error(
                        "Enter medicine name."
                    )

                else:

                    db = get_db()

                    db.execute(
                        """
                        INSERT INTO medicines
                        (user_id, medicine_name,
                         dosage, frequency,
                         medicine_time,
                         start_date, end_date,
                         status, notes)
                        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                        """,
                        (
                            users[person],
                            medicine_name,
                            dosage,
                            frequency,
                            str(medicine_time),
                            str(start_date),
                            str(end_date),
                            "Active",
                            notes
                        )
                    )

                    db.commit()
                    db.close()

                    st.success(
                        "Medicine added!"
                    )

        st.divider()

        db = get_db()

        medicines = pd.read_sql_query(
            """
            SELECT
                m.id,
                u.name AS person,
                m.medicine_name,
                m.dosage,
                m.frequency,
                m.medicine_time,
                m.start_date,
                m.end_date,
                m.status
            FROM medicines m
            LEFT JOIN users u
            ON m.user_id=u.id
            ORDER BY m.medicine_time
            """,
            db
        )

        db.close()

        st.dataframe(
            medicines,
            hide_index=True,
            use_container_width=True
        )


# ============================================================
# REMINDERS
# ============================================================

elif page == "🔔 Reminders":

    st.header("🔔 Reminder Management")

    users = get_user_options()

    if not users:

        st.warning(
            "Create an elderly profile first."
        )

    else:

        with st.form("reminder_form"):

            person = st.selectbox(
                "Elderly Person",
                list(users.keys())
            )

            title = st.text_input(
                "Reminder",
                placeholder="Take medicine"
            )

            reminder_date = st.date_input(
                "Date",
                date.today()
            )

            reminder_time = st.time_input(
                "Time",
                time(20, 0)
            )

            notes = st.text_area(
                "Notes"
            )

            submit = st.form_submit_button(
                "➕ Add Reminder"
            )

            if submit:

                if not title.strip():

                    st.error(
                        "Enter a reminder."
                    )

                else:

                    db = get_db()

                    db.execute(
                        """
                        INSERT INTO reminders
                        (user_id, title,
                         reminder_date,
                         reminder_time,
                         notes, status,
                         created_at)
                        VALUES (?, ?, ?, ?, ?, ?, ?)
                        """,
                        (
                            users[person],
                            title,
                            str(reminder_date),
                            str(reminder_time),
                            notes,
                            "Pending",
                            datetime.now().strftime(
                                "%Y-%m-%d %H:%M:%S"
                            )
                        )
                    )

                    db.commit()
                    db.close()

                    st.success(
                        "Reminder added!"
                    )

        st.divider()

        db = get_db()

        reminders = pd.read_sql_query(
            """
            SELECT
                r.id,
                u.name AS person,
                r.title,
                r.reminder_date,
                r.reminder_time,
                r.notes,
                r.status
            FROM reminders r
            LEFT JOIN users u
            ON r.user_id=u.id
            ORDER BY r.reminder_date,
                     r.reminder_time
            """,
            db
        )

        db.close()

        st.dataframe(
            reminders,
            hide_index=True,
            use_container_width=True
        )

        if not reminders.empty:

            reminder_id = st.selectbox(
                "Select reminder",
                reminders["id"].tolist()
            )

            if st.button(
                "✅ Mark Completed"
            ):

                db = get_db()

                db.execute(
                    """
                    UPDATE reminders
                    SET status='Completed'
                    WHERE id=?
                    """,
                    (reminder_id,)
                )

                db.commit()
                db.close()

                st.success(
                    "Reminder completed."
                )

                st.rerun()


# ============================================================
# ASSISTANCE REQUESTS
# ============================================================

elif page == "🤝 Assistance Requests":

    st.header("🤝 Assistance Requests")

    users = get_user_options()

    if not users:

        st.warning(
            "Create an elderly profile first."
        )

    else:

        with st.form("request_form"):

            person = st.selectbox(
                "Elderly Person",
                list(users.keys())
            )

            request_type = st.selectbox(
                "Request Type",
                [
                    "Medicine",
                    "Doctor Appointment",
                    "Transportation",
                    "Grocery",
                    "Food",
                    "Home Repair",
                    "Companionship",
                    "Technology Help",
                    "Other"
                ]
            )

            priority = st.selectbox(
                "Priority",
                [
                    "Low",
                    "Medium",
                    "High",
                    "Emergency"
                ]
            )

            description = st.text_area(
                "Describe the requirement *"
            )

            submit = st.form_submit_button(
                "📨 Submit Request"
            )

            if submit:

                if not description.strip():

                    st.error(
                        "Please describe the requirement."
                    )

                else:

                    db = get_db()

                    db.execute(
                        """
                        INSERT INTO requests
                        (user_id,
                         request_type,
                         description,
                         priority,
                         status,
                         created_at)
                        VALUES (?, ?, ?, ?, ?, ?)
                        """,
                        (
                            users[person],
                            request_type,
                            description,
                            priority,
                            "New",
                            datetime.now().strftime(
                                "%Y-%m-%d %H:%M:%S"
                            )
                        )
                    )

                    db.commit()
                    db.close()

                    st.success(
                        "Assistance request submitted!"
                    )


# ============================================================
# EMERGENCY
# ============================================================

elif page == "🆘 Emergency Help":

    st.header("🆘 Emergency Help")

    st.warning(
        "This prototype records an emergency alert "
        "in the database. It does not automatically "
        "call emergency services."
    )

    users = get_user_options()

    if not users:

        st.warning(
            "Create an elderly profile first."
        )

    else:

        with st.form("emergency_form"):

            person = st.selectbox(
                "Elderly Person",
                list(users.keys())
            )

            alert_type = st.selectbox(
                "Alert Type",
                [
                    "Immediate Assistance",
                    "Medical Assistance",
                    "Possible Fall / Accident",
                    "Unable to Contact Person",
                    "Other"
                ]
            )

            description = st.text_area(
                "What happened?"
            )

            submit = st.form_submit_button(
                "🚨 CREATE EMERGENCY ALERT"
            )

            if submit:

                db = get_db()

                db.execute(
                    """
                    INSERT INTO alerts
                    (user_id,
                     alert_type,
                     description,
                     status,
                     created_at)
                    VALUES (?, ?, ?, ?, ?)
                    """,
                    (
                        users[person],
                        alert_type,
                        description,
                        "New",
                        datetime.now().strftime(
                            "%Y-%m-%d %H:%M:%S"
                        )
                    )
                )

                db.commit()
                db.close()

                st.error(
                    "🚨 Emergency alert recorded."
                )


# ============================================================
# ACTIVITY MONITOR
# ============================================================

elif page == "🤖 Activity Monitor":

    st.header("🤖 Activity Monitor")

    st.write(
        "Use your trained ML model with HAR70+ "
        "sensor data."
    )

    source = st.radio(
        "Select data source",
        [
            "Demo Data",
            "Upload CSV"
        ],
        horizontal=True
    )

    if source == "Demo Data":

        if not DEMO_PATH.exists():

            st.error(
                "demo_sensor_data.csv not found."
            )

            st.stop()

        data = pd.read_csv(
            DEMO_PATH
        )

    else:

        uploaded = st.file_uploader(
            "Upload HAR70+ CSV",
            type=["csv"]
        )

        if uploaded is None:

            st.info(
                "Upload a sensor CSV."
            )

            st.stop()

        data = pd.read_csv(
            uploaded
        )

    st.success(
        f"{len(data):,} sensor rows loaded."
    )

    st.dataframe(
        data.head(10),
        use_container_width=True
    )

    users = get_user_options()

    user_id = None

    if users:

        person = st.selectbox(
            "Save activity for",
            list(users.keys())
        )

        user_id = users[person]

    if st.button(
        "🔮 Run Activity Prediction",
        type="primary"
    ):

        try:

            model = load_model()

            features = create_features(
                data
            )

            predictions = model.predict(
                features
            )

            if hasattr(
                model,
                "predict_proba"
            ):

                confidence = (
                    model.predict_proba(
                        features
                    ).max(axis=1) * 100
                )

            else:

                confidence = np.repeat(
                    np.nan,
                    len(predictions)
                )

            result = pd.DataFrame({

                "Window":
                    range(
                        1,
                        len(predictions) + 1
                    ),

                "Activity":
                    [
                        LABEL_NAMES.get(
                            int(x),
                            f"Class {x}"
                        )
                        for x in predictions
                    ],

                "Confidence (%)":
                    np.round(
                        confidence,
                        2
                    )
            })

            st.session_state[
                "activity_result"
            ] = result

            if user_id:

                save_activity_results(
                    user_id,
                    result
                )

            st.success(
                "Activity prediction completed!"
            )

        except Exception as e:

            st.error(
                f"Prediction failed: {e}"
            )

    if "activity_result" in st.session_state:

        result = st.session_state[
            "activity_result"
        ]

        st.subheader(
            "Prediction Results"
        )

        st.dataframe(
            result,
            hide_index=True,
            use_container_width=True
        )

        col1, col2 = st.columns(2)

        col1.metric(
            "Windows Analysed",
            len(result)
        )

        col2.metric(
            "Average Confidence",
            f"{result['Confidence (%)'].mean():.2f}%"
        )

        st.subheader(
            "Activity Distribution"
        )

        st.bar_chart(
            result["Activity"].value_counts()
        )


# ============================================================
# ACTIVITY HISTORY
# ============================================================

elif page == "📊 Activity History":

    st.header("📊 Activity History")

    users = get_user_options()

    if not users:

        st.info(
            "Create an elderly profile first."
        )

    else:

        person = st.selectbox(
            "Select elderly person",
            list(users.keys())
        )

        user_id = users[person]

        db = get_db()

        history = pd.read_sql_query(
            """
            SELECT
                activity,
                confidence,
                window_no,
                created_at
            FROM activity_results
            WHERE user_id=?
            ORDER BY id DESC
            LIMIT 500
            """,
            db,
            params=(user_id,)
        )

        db.close()

        if history.empty:

            st.info(
                "No activity history yet."
            )

        else:

            st.dataframe(
                history,
                hide_index=True,
                use_container_width=True
            )

            st.subheader(
                "Activity Summary"
            )

            st.bar_chart(
                history["activity"].value_counts()
            )

            st.metric(
                "Average Confidence",
                f"{history['confidence'].mean():.2f}%"
            )


# ============================================================
# REQUEST MANAGEMENT
# ============================================================

elif page == "📋 Request Management":

    st.header("📋 Request Management")

    db = get_db()

    requests = pd.read_sql_query(
        """
        SELECT
            r.id,
            u.name AS person,
            r.request_type,
            r.description,
            r.priority,
            r.status,
            r.created_at
        FROM requests r
        LEFT JOIN users u
        ON r.user_id=u.id
        ORDER BY r.id DESC
        """,
        db
    )

    db.close()

    if requests.empty:

        st.info(
            "No assistance requests yet."
        )

    else:

        col1, col2 = st.columns(2)

        with col1:

            priority = st.multiselect(
                "Priority",
                [
                    "Low",
                    "Medium",
                    "High",
                    "Emergency"
                ],
                default=[
                    "Low",
                    "Medium",
                    "High",
                    "Emergency"
                ]
            )

        with col2:

            status = st.multiselect(
                "Status",
                [
                    "New",
                    "In Progress",
                    "Resolved",
                    "Closed"
                ],
                default=[
                    "New",
                    "In Progress",
                    "Resolved",
                    "Closed"
                ]
            )

        filtered = requests[
            requests["priority"].isin(priority)
            &
            requests["status"].isin(status)
        ]

        st.dataframe(
            filtered,
            hide_index=True,
            use_container_width=True
        )

        if not filtered.empty:

            request_id = st.selectbox(
                "Request ID",
                filtered["id"].tolist()
            )

            new_status = st.selectbox(
                "New Status",
                [
                    "New",
                    "In Progress",
                    "Resolved",
                    "Closed"
                ]
            )

            if st.button(
                "Update Request"
            ):

                db = get_db()

                db.execute(
                    """
                    UPDATE requests
                    SET status=?
                    WHERE id=?
                    """,
                    (
                        new_status,
                        request_id
                    )
                )

                db.commit()
                db.close()

                st.success(
                    "Request updated."
                )

                st.rerun()


# ============================================================
# FOOTER
# ============================================================

st.divider()

st.caption(
    "ElderCare Assist V2 | "
    "Python + Streamlit + SQLite + Machine Learning"
)
'''

# Create the Streamlit file in the user's actual project folder
app_path = Path(r"C:\Users\eshit\eldercare_v2.py")

app_path.write_text(
    APP_CODE,
    encoding="utf-8"
)

print("ElderCare V2 created successfully!")
print(app_path)

ElderCare V2 created successfully!
C:\Users\eshit\eldercare_v2.py


In [18]:
import os
print(
     "V2 app:",
    os.path.exists(r"C:\Users\eshit\eldercare_v2.py")
)
print(
    "ML model:",
    os.path.exists(r"C:\Users\eshit\eldercare_activity_model.joblib")
)
print(
    "Demo data:",
    os.path.exists(r"C:\Users\eshit\demo_sensor_data.csv")
)

V2 app: True
ML model: True
Demo data: True


In [19]:
import subprocess
import sys
app_path = r"C:\Users\eshit\eldercare_v2.py"
process = subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    app_path
])
print("ElderCare Assist V2 is starting...")

ElderCare Assist V2 is starting...
